# 🍅 TomatoHealth AI — Model Training

Notebook ini akan:
1. Download dataset PlantVillage dari Kaggle
2. Preprocessing & augmentasi data
3. Training MobileNetV2 (2 tahap)
4. Evaluasi (accuracy, confusion matrix)
5. Export `tomato_model.h5`

**Pastikan GPU aktif:** Runtime → Change runtime type → T4 GPU

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q kaggle matplotlib seaborn scikit-learn

## Cell 2 — Setup Kaggle API Token
Ganti token di bawah dengan token kamu dari kaggle.com/settings/api

In [ ]:
import os

# GANTI dengan token Kaggle kamu
KAGGLE_TOKEN = 'KGAT_daa331266da7750be396bf5a0bc58ba3'

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

print('✅ Kaggle API token configured')

## Cell 3 — Download Dataset PlantVillage

In [ ]:
!kaggle datasets download -d abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d /content/dataset
print('✅ Dataset downloaded & extracted')

## Cell 4 — Filter Folder Tomato Saja

In [ ]:
import shutil

SOURCE_DIR = '/content/dataset/plantvillage dataset/color'
TOMATO_DIR = '/content/tomato_data'
os.makedirs(TOMATO_DIR, exist_ok=True)

tomato_folders = sorted([f for f in os.listdir(SOURCE_DIR) if 'Tomato' in f])

print(f'Ditemukan {len(tomato_folders)} kelas tomat:\n')
for i, folder in enumerate(tomato_folders):
    src = os.path.join(SOURCE_DIR, folder)
    dst = os.path.join(TOMATO_DIR, folder)
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
    count = len(os.listdir(dst))
    print(f'  {i}: {folder} ({count} foto)')

## Cell 5 — Preprocessing & Data Generator

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    TOMATO_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_generator = val_datagen.flow_from_directory(
    TOMATO_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

print(f'\nTraining samples: {train_generator.samples}')
print(f'Validation samples: {val_generator.samples}')
print(f'Jumlah kelas: {train_generator.num_classes}')
print(f'\nLabel mapping:')
for name, idx in sorted(train_generator.class_indices.items(), key=lambda x: x[1]):
    print(f'  {idx}: {name}')

## Cell 6 — Build Model (MobileNetV2 + Custom Classifier)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

NUM_CLASSES = train_generator.num_classes

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Cell 7 — Tahap 1: Training Classifier (5 epoch, base frozen)

In [ ]:
print('=' * 50)
print('TAHAP 1: Training classifier saja')
print('=' * 50)

history1 = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator,
    verbose=1
)

## Cell 8 — Tahap 2: Fine-tune 30 Layer Terakhir (10 epoch)

In [ ]:
print('=' * 50)
print('TAHAP 2: Fine-tuning 30 layer terakhir')
print('=' * 50)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    verbose=1
)

## Cell 9 — Evaluasi: Confusion Matrix & Accuracy Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

val_generator.reset()
preds = model.predict(val_generator, verbose=1)
y_pred = np.argmax(preds, axis=1)
y_true = val_generator.classes

class_names = list(val_generator.class_indices.keys())
short_names = [c.replace('Tomato___', '').replace('Tomato__', '').replace('_', ' ') for c in class_names]

print('\nCLASSIFICATION REPORT')
print('=' * 50)
print(classification_report(y_true, y_pred, target_names=short_names))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=short_names, yticklabels=short_names)
plt.title('Confusion Matrix - TomatoHealth AI', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Training History
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
all_acc = history1.history['accuracy'] + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss = history1.history['loss'] + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']

ax1.plot(all_acc, label='Train')
ax1.plot(all_val_acc, label='Validation')
ax1.axvline(x=4.5, color='gray', linestyle='--', label='Fine-tune start')
ax1.set_title('Accuracy')
ax1.legend()

ax2.plot(all_loss, label='Train')
ax2.plot(all_val_loss, label='Validation')
ax2.axvline(x=4.5, color='gray', linestyle='--', label='Fine-tune start')
ax2.set_title('Loss')
ax2.legend()
plt.tight_layout()
plt.show()

val_loss, val_acc = model.evaluate(val_generator)
print(f'\n✅ Final Validation Accuracy: {val_acc:.4f}')

## Cell 10 — Label Map untuk Backend

In [ ]:
label_map_indo = {
    'healthy': 'Sehat',
    'Bacterial_spot': 'Bercak Bakteri (Bacterial Spot)',
    'Early_blight': 'Bercak Daun Awal (Early Blight)',
    'Late_blight': 'Bercak Daun Akhir (Late Blight)',
    'Leaf_Mold': 'Jamur Daun (Leaf Mold)',
    'Septoria_leaf_spot': 'Bercak Septoria (Septoria Leaf Spot)',
    'Spider_mites Two-spotted_spider_mite': 'Tungau Laba-laba (Spider Mites)',
    'Target_Spot': 'Bercak Target (Target Spot)',
    'Tomato_Yellow_Leaf_Curl_Virus': 'Virus Keriting Kuning (Yellow Leaf Curl)',
    'Tomato_mosaic_virus': 'Virus Mosaik (Mosaic Virus)',
}

print('LABEL_MAP untuk backend/ml/label_map.py:\n')
print('LABEL_MAP = {')
for class_name, idx in sorted(val_generator.class_indices.items(), key=lambda x: x[1]):
    short = class_name.replace('Tomato___', '').replace('Tomato__', '')
    indo = label_map_indo.get(short, short)
    print(f'    {idx}: "{indo}",')
print('}')

## Cell 11 — Export & Download Model

In [ ]:
# Simpan model
model.save('/content/tomato_model.h5')
size_mb = os.path.getsize('/content/tomato_model.h5') / (1024 * 1024)
print(f'✅ Model saved: tomato_model.h5 ({size_mb:.1f} MB)')

# Download ke laptop
from google.colab import files
files.download('/content/tomato_model.h5')

print('\n' + '=' * 50)
print('SELESAI!')
print('Taruh file di: backend/ml/models/tomato_model.h5')
print('=' * 50)